# 06 · Power BI Data Prep

Power BI Desktop doesn't have a built-in SQLite connector without an extra
ODBC driver install, but it reads **Parquet natively** (`Get Data > Parquet`
in current Power BI Desktop builds). This notebook gathers every table the
dashboard needs — dimensions, facts, and every Phase 3/4/5 output — into one
folder: `data/powerbi/`. Point Power BI at that single folder and every
table below is ready to import and relate.

Two tables are intentionally *not* exported because they're redundant with
`reco_replenishment` (same sku x warehouse grain, superset of columns —
KPIs + health score + reorder/transfer recommendation, all in one table).

In [1]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/inventory-intelligence-gsc')
except ImportError:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
import powerbi_prep as pp

counts = pp.build_powerbi_folder(PROJECT_ROOT)
print(f"Wrote {len(counts)} tables to data/powerbi/:")
for name, n in counts.items():
    print(f"  {name}: {n:,} rows")

Wrote 14 tables to data/powerbi/:
  dim_sku: 1,540 rows
  dim_warehouse: 19 rows
  dim_date: 731 rows
  dim_replenishment_policy: 2,035 rows
  fact_inventory: 1,487,585 rows
  fact_demand: 1,487,585 rows
  fact_orders: 17,119 rows
  kpi_monthly_diagnostic: 1,224 rows
  predictive_forecast_test: 16,280 rows
  reco_replenishment: 2,035 rows
  reco_transfers: 2 rows
  reco_unmet_shortage: 54 rows
  reco_transfer_cost_matrix: 342 rows
  dim_warehouse_coords: 19 rows


## Next: open Power BI Desktop

1. `Get Data` → `Folder` → point at `data/powerbi/` in your project folder (or import each `.parquet` file individually via `Get Data` → `Parquet`)
2. Load all 13 tables
3. Follow `docs/POWERBI_BUILD_GUIDE.md` for the relationships and the 7 pages, and `docs/POWERBI_DAX_REFERENCE.md` for every measure, copy-paste ready

## Table reference

| Table | Grain | Role |
|---|---|---|
| `dim_sku` | 1 row / SKU | Dimension |
| `dim_warehouse` | 1 row / warehouse | Dimension |
| `dim_date` | 1 row / day | Dimension (date table) |
| `dim_replenishment_policy` | 1 row / SKU x warehouse | Dimension (static policy) |
| `fact_inventory` | 1 row / SKU x warehouse x day | Fact (daily on-hand/backorder) |
| `fact_demand` | 1 row / SKU x warehouse x day | Fact (daily demand) |
| `fact_orders` | 1 row / order | Fact (replenishment orders placed) |
| `kpi_monthly_diagnostic` | 1 row / month x warehouse x ABC class | Fact (inventory value trend) |
| `predictive_forecast_test` | 1 row / SKU x warehouse x test week | Fact (forecast vs. actual) |
| `reco_replenishment` | 1 row / SKU x warehouse | **Main snapshot** — KPIs, health score, stockout risk, reorder & excess flags, all current |
| `reco_transfers` | 1 row / recommended transfer | Fact (Phase 5 LP output) |
| `reco_unmet_shortage` | 1 row / SKU x warehouse with unmet shortage | Fact (Phase 5) |
| `reco_transfer_cost_matrix` | 1 row / warehouse pair | Reference (transfer cost model) |
| `dim_warehouse_coords` | 1 row / warehouse | Dimension — **illustrative** map coordinates only (region-center + deterministic jitter; the source data has no real facility geolocation) |